In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import mlflow
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline

In [2]:
load_dotenv()
df_path = os.getenv("DATASET_PATH")
file_path = os.path.join(df_path, "data.csv")
df = pd.read_csv(file_path)
pd.set_option('display.max_columns', None)

In [3]:
df.isna().sum()[df.isna().sum()>0]

Unnamed: 32    569
dtype: int64

In [4]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [5]:
df = df.drop("Unnamed: 32", axis=1)

In [6]:
df['diagnosis'].value_counts(normalize=True)

diagnosis
B    0.627417
M    0.372583
Name: proportion, dtype: float64

In [7]:
X = df.drop(['id', 'diagnosis'], axis=1)
y = df['diagnosis']
y = y.map({"B":0, "M":1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [8]:
models = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0),
    "Sklearn_HistGB": HistGradientBoostingClassifier(random_state=42),
    "RandomFores": RandomForestClassifier(random_state=42)}

results = []

In [15]:
mlflow.set_experiment("5 Models Comparison")
mlflow.set_tracking_uri("http://127.0.0.1:5000")

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        probs = pipe.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        roc_auc = roc_auc_score(y_test, probs)
        f1 = f1_score(y_test, preds)

        results.append({"model":model_name, "accuracy":acc, "roc_auc_score":roc_auc, "f1_score":f1})
        mlflow.log_params(pipe.get_params())
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc_score", roc_auc)

        mlflow.sklearn.log_model(pipe, artifact_path="model_pipeline", serialization_format="pickle")

2026/07/29 10:40:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/29 10:40:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/4/runs/df1836a5453246de939bbfff49a8ada5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/07/29 10:40:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/29 10:40:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/4/runs/b0085e40e71c4b698864a4faeeec5de3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/07/29 10:40:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/29 10:40:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run CatBoost at: http://127.0.0.1:5000/#/experiments/4/runs/103fc9ffd82e4619aa442bd212896d5b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/07/29 10:41:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/29 10:41:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Sklearn_HistGB at: http://127.0.0.1:5000/#/experiments/4/runs/e8da187d1593416bbfa100464e15f2fd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/07/29 10:41:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/29 10:41:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomFores at: http://127.0.0.1:5000/#/experiments/4/runs/28eb216a2e28423cb65b49b9361e2ad0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
